In [2]:
!pip install transformers datasets torch scikit-learn -q

In [3]:
from datasets import load_dataset

dataset = load_dataset("ahmedheakl/resume-atlas")
print(dataset)
print(dataset["train"][0])

README.md:   0%|          | 0.00/215 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/53.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13389 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Category', 'Text'],
        num_rows: 13389
    })
})
{'Category': 'Accountant', 'Text': 'education omba executive leadership university texas 20162018 bachelor science accounting richland college 20052008 training certifications certified management accountant cma certified financial modeling valuation analyst compliance antimoney laundering 092016 american institute banking certified public account cpa lean six sigma green belt certified trade products financial regulations 082016 american institute banking achievements speaker bringing leader within 082019 successfully presented empowering speech leadership 500 participants speaker dallas convention cpas 032019 successfully delivered seminar 3k cpas convention guests teaching experience online teacher udemy 2017 taught online accounting nonaccountant course udemy similar online teaching platforms developed effective teaching modules materials curriculum target students took feed

In [4]:
import pandas as pd

df = dataset["train"].to_pandas()
print(f"Total samples: {len(df)}")
print(f"\nCategories ({df['Category'].nunique()} total):")
print(df["Category"].value_counts())

Total samples: 13389

Categories (43 total):
Category
Education                    410
Electrical Engineering       384
Mechanical Engineer          384
Consultant                   368
Sales                        364
Civil Engineer               364
Management                   361
Human Resources              360
Digital Media                358
Accountant                   350
Java Developer               348
Operations Manager           345
Building and Construction    345
Testing                      344
Architecture                 344
Aviation                     340
Business Analyst             340
Finance                      339
SQL Developer                338
Public Relations             337
Health and Fitness           332
Arts                         332
Network Security Engineer    330
DotNet Developer             329
Apparel                      320
Banking                      314
Automobile                   313
Web Designing                309
SAP Developer         

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer
import torch

# Encode labels
le = LabelEncoder()
df["label"] = le.fit_transform(df["Category"])

# Trim text
df["input"] = df["Text"].str[:2000]

# Split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df["label"])

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Classes: {len(le.classes_)}")

# Tokenize
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(texts):
    return tokenizer(list(texts), truncation=True, padding=True, max_length=512, return_tensors="pt")

train_enc = tokenize(train_df["input"])
val_enc   = tokenize(val_df["input"])
print("Tokenization done ")

Train: 9639 | Val: 1072 | Test: 2678
Classes: 43


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenization done 


In [6]:
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification

class ResumeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = ResumeDataset(train_enc, train_df["label"])
val_dataset   = ResumeDataset(val_enc,   val_df["label"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=43
)
model = model.to(device)

print(f"Device: {device} ")
print(f"Model loaded with 43 output classes ")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda 
Model loaded with 43 output classes 


In [ ]:
from torch.optim import AdamW
from torch.utils.data import DataLoader
import os

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
optimizer    = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()

    avg_loss = total_loss / len(train_loader)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            preds = model(input_ids=input_ids, attention_mask=attention_mask).logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {correct/total*100:.2f}%")

# Save immediately after training
os.makedirs("resume_model", exist_ok=True)
model.save_pretrained("resume_model")
tokenizer.save_pretrained("resume_model")
with open("resume_model/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("\nTraining done & model saved locally ")

In [8]:
import pickle

with open("resume_model/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Label encoder saved ")

Label encoder saved 


In [9]:
from sklearn.metrics import classification_report

test_enc     = tokenize(test_df["input"])
test_dataset = ResumeDataset(test_enc, test_df["label"])
test_loader  = DataLoader(test_dataset, batch_size=32)

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        preds = model(input_ids=input_ids, attention_mask=attention_mask).logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=le.classes_))

                           precision    recall  f1-score   support

               Accountant       0.93      0.91      0.92        70
                 Advocate       0.89      0.97      0.93        58
              Agriculture       0.79      0.85      0.82        59
                  Apparel       0.95      0.81      0.87        64
             Architecture       0.88      0.65      0.75        69
                     Arts       0.88      0.88      0.88        66
               Automobile       0.71      0.57      0.63        63
                 Aviation       0.96      0.97      0.96        68
                      BPO       0.89      0.60      0.72        40
                  Banking       0.94      0.95      0.94        63
               Blockchain       0.00      0.00      0.00         9
Building and Construction       0.82      0.87      0.85        69
         Business Analyst       0.88      0.88      0.88        68
           Civil Engineer       0.95      0.97      0.96     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
